In [ ]:
import matplotlib.pyplot as plt
import imageio
import numpy as np
import pandas as pd

import sys
sys.path.append('./src')
import warnings
warnings.filterwarnings("ignore")

from data import PPCI

In [ ]:
dataset = PPCI(encoder = "dino",
               token = "class",
               task = "or",
               split_criteria = "position_easy",
               environment = "supervised",
               batch_size = 256,
               num_proc = 4,
               verbose = True,
               data_dir = 'data/istant_lq',
               results_dir = 'results/istant_lq')
#dataset.plot_out_distribution()
# dataset.train(add_pred_env="supervised", 
#               hidden_layers = 2,
#               hidden_nodes = 256,
#               batch_size = 256,
#               lr = 0.0005,
#               seed = 4,
#               num_epochs=10,
#               save = False,
#               verbose=True)
# dataset.visualize(k=8, save=True)
# result_tr_y = dataset.evaluate(color="yellow", train=True, verbose=False)
# result_tr_b = dataset.evaluate(color="blue", train=True, verbose=False)
# result_val_y = dataset.evaluate(color="yellow", train=False, verbose=False)
# result_val_b = dataset.evaluate(color="blue", train=False, verbose=False)

In [ ]:
dataset.evaluate()

In [ ]:
exp = 0
pos = 6
frame = 730

frame_id = ((dataset.supervised["source_data"]["experiment"] == exp) & (dataset.supervised["source_data"]["position"] == pos) & (dataset.supervised["source_data"]["frame"] == frame)).nonzero(as_tuple=True)[0][0].item()
img = dataset.supervised["source_data"][frame_id]["image"] # shape 3, 770, 770
print(dataset.supervised["Y"][frame_id])
print(dataset.supervised["Y"][frame_id])
# remove ticks
plt.axis('off')
plt.imshow(img.permute(1, 2, 0));

In [ ]:
np.where(dataset.supervised["Y"][(dataset.supervised["source_data"]["experiment"] == 0) & (dataset.supervised["source_data"]["position"] == 6)]==1)

In [ ]:
# get .GIF
fps = 2
speed = 2

# treatment exp
exp = 0
pos = 6
frames = list(range(330,380))
# # control exp
# exp = 1
# pos = 4
# frames = list(range(300,340))

imgs = []
descs1 = []
descs2 = []
colors = []
for frame in frames:
    frame_id = ((dataset.supervised["source_data"]["experiment"] == exp) & (dataset.supervised["source_data"]["position"] == pos) & (dataset.supervised["source_data"]["frame"] == frame)).nonzero(as_tuple=True)[0][0].item()
    img = dataset.supervised["source_data"][frame_id]["image"].permute(1, 2, 0).numpy().astype(np.uint8) # shape 770, 770, 3
    W = dataset.supervised["W"][frame_id]
    Y = dataset.supervised["Y"][frame_id]
    T = dataset.supervised["T"][frame_id]
    Y_hat_ = dataset.supervised["Y_hat"][frame_id]
    Y_hat = Y_hat_.round()
    if Y>Y_hat:
        comment = "(False Positive)"
    elif Y<Y_hat:
        comment = "(False Negative)"
    else:
        comment = ""
    treatment = "Treated Group" if T==2 else "Control Group"

    imgs.append(img)
    behaviour = "Grooming" if Y==1 else "No Action"
    descs1.append(f"{treatment}\nBatch: {exp+1}, Position: {pos}, Time: {frame//(60*fps)}m{round((frame%(60*fps))/fps)}s")
    descs2.append(f"{'Grooming' if Y_hat==1 else 'No Action'} ({(Y_hat*Y_hat_+(1-Y_hat)*(1-Y_hat_))*100:.2f}%)\n{comment}")
    colors.append('g' if Y==Y_hat else 'r')

bs = 5
us = 3
frames = []
for img, desc1, desc2, color in zip(imgs, descs1, descs2, colors):
    height, width = img.shape[:2]
    fig_height = height / 100 
    fig_width = width / 100
    fig = plt.figure(figsize=(fig_width, fig_height + bs + us))
    ax = fig.add_axes([0, bs / (fig_height + bs), 1, fig_height / (fig_height + bs)])
  
    ax.imshow(img) 
    ax.axis('off')
    plt.figtext(0.5, 0.952, desc1, ha="center", fontsize=20)
    plt.figtext(0.5, 0.02, desc2, ha="center", fontsize=30, color=color)

    fig.canvas.draw()
    frame = np.frombuffer(fig.canvas.tostring_rgb(), dtype='uint8')
    frame = frame.reshape(fig.canvas.get_width_height()[::-1] + (3,))
    frames.append(frame)
    plt.close(fig)

# Save the frames as a gif
imageio.mimsave(f'img/B{exp+1}P{pos}T{T}.gif', frames, fps=speed, loop=0)

In [ ]:
# get .GIF only Y
fps = 2
speed = 2

# # treatment exp
exp = 0
pos = 6
frames = list(range(330,380))
# control exp
# exp = 1
# pos = 4
# frames = list(range(300,340))

imgs = []
descs1 = []
descs2 = []
colors = []
for frame in frames:
    frame_id = ((dataset.supervised["source_data"]["experiment"] == exp) & (dataset.supervised["source_data"]["position"] == pos) & (dataset.supervised["source_data"]["frame"] == frame)).nonzero(as_tuple=True)[0][0].item()
    img = dataset.supervised["source_data"][frame_id]["image"].permute(1, 2, 0).numpy().astype(np.uint8) # shape 770, 770, 3
    W = dataset.supervised["W"][frame_id]
    Y = dataset.supervised["Y"][frame_id]
    T = dataset.supervised["T"][frame_id]
    treatment = "Treated Group" if T==2 else "Control Group"

    imgs.append(img)
    behaviour = "Grooming" if Y==1 else "No Action"
    descs1.append(f"{treatment}\nBatch: {exp+1}, Position: {pos}, Time: {frame//(60*fps)}m{round((frame%(60*fps))/fps)}s")
    descs2.append(f"{behaviour}")

bs = 5
us = 3
frames = []
for img, desc1, desc2 in zip(imgs, descs1, descs2):
    height, width = img.shape[:2]
    fig_height = height / 100 
    fig_width = width / 100
    fig = plt.figure(figsize=(fig_width, fig_height + bs + us))
    ax = fig.add_axes([0, bs / (fig_height + bs), 1, fig_height / (fig_height + bs)])
  
    ax.imshow(img) 
    ax.axis('off')
    plt.figtext(0.5, 0.952, desc1, ha="center", fontsize=20)
    plt.figtext(0.5, 0.02, desc2, ha="center", fontsize=30)

    fig.canvas.draw()
    frame = np.frombuffer(fig.canvas.tostring_rgb(), dtype='uint8')
    frame = frame.reshape(fig.canvas.get_width_height()[::-1] + (3,))
    frames.append(frame)
    plt.close(fig)

# Save the frames as a gif
imageio.mimsave(f'img/B{exp+1}P{pos}T{T}Y.gif', frames, fps=speed, loop=0)

In [ ]:
exp = 0
pos = 6
frame = 450

frame_id = ((dataset.supervised["source_data"]["experiment"] == exp) & (dataset.supervised["source_data"]["position"] == pos) & (dataset.supervised["source_data"]["frame"] == frame)).nonzero(as_tuple=True)[0][0].item()
img1 = dataset.supervised["source_data"][frame_id]["image"] # shape 3, 770, 770
print(dataset.supervised["Y"][frame_id])

exp = 0
pos = 6
frame = 720

frame_id = ((dataset.supervised["source_data"]["experiment"] == exp) & (dataset.supervised["source_data"]["position"] == pos) & (dataset.supervised["source_data"]["frame"] == frame)).nonzero(as_tuple=True)[0][0].item()
img2 = dataset.supervised["source_data"][frame_id]["image"] # shape 3, 770, 770
print(dataset.supervised["Y"][frame_id])

# plot the 2 images
fig, ax = plt.subplots(1, 2)
ax[0].imshow(img1.permute(1, 2, 0))
ax[1].imshow(img2.permute(1, 2, 0))
# remove ticks
for a in ax:
    a.set_xticks([])
    a.set_yticks([])
# attach the plot closer
plt.show()

In [ ]:
dataset.evaluate(color="blue", verbose=False)

In [ ]:
from causal import compute_ate
compute_ate(dataset.supervised["Y_hat"], 
            dataset.supervised["T"], 
            dataset.supervised["W"], 
            method="ead", 
            color="blue")

In [ ]:
# dataset = PPCI()
# dataset.plot_out_distribution()
# dataset.train()
# dataset.visualize()
# dataset.evaluate()

## Post-Processing

In [ ]:
exp = (dataset.supervised["source_data"]["experiment"]==4)
pos = (dataset.supervised["source_data"]["position"]==1)
filter = (exp & pos).nonzero().squeeze()
y = dataset.supervised["Y"][filter][:,0].detach()
y_hat = dataset.supervised["Y_hat"][filter][:,0].detach()
y_pred = y_hat.round()

plt.scatter(range(len(filter)), y_hat, s=1, c="blue", alpha=0.5, label="y_probs")
plt.scatter(range(len(filter)),y_pred-(-1)**y_pred.detach()*0.04, s=1, c="red", alpha=0.5, label="y_pred")
plt.scatter(range(len(filter)), y-(-1)**y.detach()*0.02, s=1, c="green", alpha=0.5, label="y")
plt.legend()
plt.show()

In [ ]:
frame = (dataset.supervised["source_data"]["frame"]==2220)
idx = (exp & pos & frame).nonzero().item()
img = dataset.supervised["source_data"][idx]["image"]
outcome = dataset.supervised["source_data"][idx]["outcome"]

img = img.permute(1, 2, 0)
plt.title(f"Y2F: {int(outcome[0])}, B2F: {int(outcome[1])}")
plt.imshow(img);